## **2. Feature Engineering & Model Training**

This notebook covers:
- Loading prepared train/test data
- Building text + numeric features
- Training multiple models
- Cross-validation
- Model comparison
- Saving the best model pipeline

In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import cross_val_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

ARTIFACT_DIR = Path("artifacts")
MODEL_DIR = Path("model_store")
MODEL_DIR.mkdir(exist_ok=True)

train_df = pd.read_csv(ARTIFACT_DIR / "train_prepared.csv")
test_df = pd.read_csv(ARTIFACT_DIR / "test_prepared.csv")

train_df.head()

,id,keyword,location,text,clean_text,tokens,text_length_chars,word_count,has_hashtag,has_mention,has_url,target
0,8902,snowstorm,"South, USA",Sassy city girl country hunk stranded in Smoky...,sassy city girl country hunk stranded in smoky...,"['sassy', 'city', 'girl', 'country', 'hunk', '...",116,14,True,False,True,1
1,472,armageddon,Worldwide,God's Kingdom (Heavenly Gov't) will rule over ...,god s kingdom heavenly gov t will rule over al...,"['god', 's', 'kingdom', 'heavenly', 'gov', 't'...",135,16,False,False,True,0
2,1448,body%20bagging,Cloud 9,Mopheme and Bigstar Johnson are a problem in t...,mopheme and bigstar johnson are a problem in t...,"['mopheme', 'and', 'bigstar', 'johnson', 'are'...",86,14,True,False,False,0
3,10407,whirlwind,Sheff/Bangor/Salamanca/Madrid,@VixMeldrew sounds like a whirlwind life!,sounds like a whirlwind life,"['sounds', 'like', 'a', 'whirlwind', 'life']",41,6,False,True,False,0
4,3137,debris,Nigeria,Malaysia confirms plane debris washed up on Re...,malaysia confirms plane debris washed up on re...,"['malaysia', 'confirms', 'plane', 'debris', 'w...",102,14,False,False,True,1


# **Feature design**

We will use:
- `clean_text` with TF-IDF word features
- simple numeric metadata:
  - `text_length_chars`
  - `word_count`
  - `has_hashtag`
  - `has_mention`
  - `has_url`

In [2]:
text_col = "clean_text"
numeric_cols = ["text_length_chars", "word_count", "has_hashtag", "has_mention", "has_url"]

X_train = train_df[[text_col] + numeric_cols]
y_train = train_df["target"]

X_test = test_df[[text_col] + numeric_cols]
y_test = test_df["target"]

In [3]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "tfidf",
            TfidfVectorizer(
                max_features=20000,
                ngram_range=(1, 2),
                min_df=2,
                max_df=0.95,
                sublinear_tf=True
            ),
            text_col
        ),
        (
            "num",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler(with_mean=False))
                ]
            ),
            numeric_cols
        )
    ],
    remainder="drop"
)

In [4]:
models = {
    "logistic_regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "linear_svc": LinearSVC(class_weight="balanced"),
    "multinomial_nb": MultinomialNB(),
    "random_forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_split=5,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced_subsample"
    )
}

In [5]:
results = []

for model_name, model in models.items():
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    cv_f1 = cross_val_score(pipe, X_train, y_train, cv=5, scoring="f1").mean()
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)

    row = {
        "model": model_name,
        "cv_f1": round(cv_f1, 4),
        "test_accuracy": round(accuracy_score(y_test, preds), 4),
        "test_precision": round(precision_score(y_test, preds), 4),
        "test_recall": round(recall_score(y_test, preds), 4),
        "test_f1": round(f1_score(y_test, preds), 4),
    }
    results.append(row)

results_df = pd.DataFrame(results).sort_values(by=["test_f1", "cv_f1"], ascending=False).reset_index(drop=True)
results_df

,model,cv_f1,test_accuracy,test_precision,test_recall,test_f1
0,linear_svc,0.7334,0.7997,0.7672,0.7661,0.7666
1,logistic_regression,0.7405,0.7945,0.7603,0.7615,0.7609
2,multinomial_nb,0.6976,0.7938,0.8602,0.6208,0.7211
3,random_forest,0.6947,0.7859,0.8293,0.6315,0.7170


In [6]:
best_model_name = results_df.loc[0, "model"]
best_model_name

'linear_svc'

In [7]:
best_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", models[best_model_name])
])

best_pipeline.fit(X_train, y_train)

joblib.dump(best_pipeline, MODEL_DIR / "best_disaster_tweet_model.joblib")
results_df.to_csv(MODEL_DIR / "model_comparison.csv", index=False)

print("Saved best model to:", MODEL_DIR / "best_disaster_tweet_model.joblib")
print("Saved comparison table to:", MODEL_DIR / "model_comparison.csv")

Saved best model to: model_store\best_disaster_tweet_model.joblib
Saved comparison table to: model_store\model_comparison.csv


## **Optional hyperparameter tuning**

Run the next cell if you want to tune the strongest baseline further.

In [8]:
# Optional tuning example for Logistic Regression
from sklearn.model_selection import GridSearchCV

logreg_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

param_grid = {
    "model__C": [0.5, 1.0, 2.0, 5.0],
    "preprocessor__tfidf__max_features": [10000, 15000, 20000],
    "preprocessor__tfidf__ngram_range": [(1, 1), (1, 2)]
}

grid = GridSearchCV(
    estimator=logreg_pipe,
    param_grid=param_grid,
    scoring="f1",
    cv=3,
    n_jobs=-1,
    verbose=1
)

# Uncomment to run tuning
# grid.fit(X_train, y_train)
# print("Best params:", grid.best_params_)
# print("Best CV F1:", grid.best_score_)
# tuned_best_model = grid.best_estimator_
# joblib.dump(tuned_best_model, MODEL_DIR / "best_tuned_disaster_tweet_model.joblib")

## **Notebook 2 Output**

Artifacts created:
- `model_store/best_disaster_tweet_model.joblib`
- `model_store/model_comparison.csv`

These will be used in evaluation and deployment.